# Qwen3-TTS 動作検証
**ランタイム設定**: ランタイム → ランタイムのタイプを変更 → **T4 GPU** を選択してから実行してください。

## 1. GPU確認

In [11]:
import torch

print(f"CUDA利用可能: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    raise RuntimeError("GPUが検出されません。ランタイムをT4 GPUに変更してください。")

CUDA利用可能: True
GPU: Tesla T4
VRAM: 14.6 GB


## 2. ライブラリのインストール

In [12]:
!pip install -q -U qwen-tts soundfile

In [13]:
import importlib, qwen_tts, soundfile
print(f"qwen-tts: OK (version: {importlib.metadata.version('qwen-tts')})")
print(f"soundfile: OK")

qwen-tts: OK (version: 0.1.1)
soundfile: OK


## 3. ボイスクローン：Base モデルをロード

初回は約3.5GBのダウンロードが発生します。

In [16]:
import torch
from qwen_tts import Qwen3TTSModel

MODEL_BASE = "Qwen/Qwen3-TTS-12Hz-1.7B-Base"

print(f"ロード中: {MODEL_BASE}")
model_base = Qwen3TTSModel.from_pretrained(
    MODEL_BASE,
    device_map="cuda:0",
    dtype=torch.bfloat16,
)
print("ロード完了！")

VRAM解放後: 3.95 GB 使用中
ロード中: Qwen/Qwen3-TTS-12Hz-1.7B-Base


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

ロード完了！


## 4. 参照音声のアップロード
- **3〜15秒**のクリアなWAVファイルを用意してください
- ノイズ・BGMなし・モノラル推奨・24kHz以上

In [17]:
from google.colab import files
import IPython.display as ipd

print("WAVファイルを選択してアップロードしてください...")
uploaded = files.upload()

if not uploaded:
    raise ValueError("ファイルがアップロードされませんでした。")

REF_AUDIO_PATH = list(uploaded.keys())[0]
print(f"アップロード完了: {REF_AUDIO_PATH}")

print("参照音声の確認:")
ipd.display(ipd.Audio(REF_AUDIO_PATH))

WAVファイルを選択してアップロードしてください...


Saving 「これをもちまして公演を終了させていただきます。...ご来場頂きまして誠にありがとうございました」.wav to 「これをもちまして公演を終了させていただきます。...ご来場頂きまして誠にありがとうございました」 (1).wav
アップロード完了: 「これをもちまして公演を終了させていただきます。...ご来場頂きまして誠にありがとうございました」 (1).wav
参照音声の確認:


## 5. バッチ生成（甘々ASMR・ささやき）
同じ声で複数テキストを生成します。プロンプトを事前計算して再利用することで高速化できます。

In [19]:
import soundfile as sf
import IPython.display as ipd
import gc, torch, glob, os, re

# ============================================================
# ★ 参照音声の書き起こしテキスト（アップロードした音声と一致させる）
# ============================================================
REF_TEXT = "ここにアップロードした音声の書き起こしを入力してください"

# ============================================================
# ★ 演技ニュアンス指定（英語で記述 → 日本語より精度が高い）
#
#  ASMR例:
#    "ASMR style, extremely soft whispering voice, intimate and gentle,
#     breathy and close-mic delivery, calm and soothing tone"
#
#  その他の例:
#    "Warm and tender voice, slow pace, slightly emotional"
#    "Energetic and cheerful, upbeat tone"
#    ""  ← 空文字にすると通常のボイスクローンのみ
# ============================================================
INSTRUCT = (
    "ASMR style, extremely soft whispering voice, intimate and gentle, "
    "breathy and close-mic delivery, calm and soothing tone"
)

# ============================================================
# ★ ASMRスクリプト
#    必要に応じてテキストを変更・追加してください
# ============================================================
TEXTS = [
    "ねえ……聞こえてる？そっと、耳を澄ませて……",
    "今日も、お疲れ様でした……ゆっくり、息を吐いて。大丈夫ですよ。",
    "もう少しだけ、そばにいますね……何も心配しなくていいです。",
    "目を閉じて……ただ、私の声だけ、聞いていて。",
    "おやすみなさい……ゆっくり、眠れますように……。",
]

# ---- 前回の出力ファイルを削除 & GPU メモリを解放 ----
for old in glob.glob("output_*.wav"):
    os.remove(old)
    print(f"削除: {old}")
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM: {torch.cuda.memory_allocated() / 1024**3:.2f} GB 使用中")

def safe_filename(text, max_len=40):
    name = re.sub(r'[\\/:*?"<>|]', "", text)
    name = re.sub(r"\s+", "_", name.strip())
    return name[:max_len] + ".wav"

# ---- ボイスプロンプトを事前計算（参照音声は1回だけ処理） ----
print("ボイスプロンプトを事前計算中...")
prompt_items = model_base.create_voice_clone_prompt(
    ref_audio=REF_AUDIO_PATH,
    ref_text=REF_TEXT,
)
print(f"プロンプト計算完了。ニュアンス: {INSTRUCT or '(なし)'}")
print("音声生成中...")

wavs, sr = model_base.generate_voice_clone(
    text=TEXTS,
    language=["Japanese"] * len(TEXTS),
    voice_clone_prompt=prompt_items,
    instruct=INSTRUCT if INSTRUCT else None,
)

output_files = []
for i, wav in enumerate(wavs):
    path = safe_filename(TEXTS[i])
    sf.write(path, wav, sr)
    output_files.append(path)
    print(f"\n--- [{i+1}] {TEXTS[i]} ---")
    print(f"保存: {path}")
    ipd.display(ipd.Audio(path))

print(f"\nバッチ生成完了！ {len(output_files)}件")

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


ボイスプロンプトを事前計算中...
プロンプト計算完了。音声生成中...

--- [1] おはようございます。今日もよろしくお願いします。 ---



--- [2] こちらは２番目の音声サンプルです。 ---



--- [3] バッチ処理により効率よく複数の音声を生成しています。 ---



バッチ生成完了！


## 6. 生成ファイルのダウンロード

In [ ]:
from google.colab import files
import glob, os

# 参照音声・アップロードファイルを除いた全 .wav をダウンロード
exclude = {os.path.basename(REF_AUDIO_PATH)} if 'REF_AUDIO_PATH' in dir() else set()

wav_files = [f for f in glob.glob("*.wav") if os.path.basename(f) not in exclude]
print(f"ダウンロード対象: {len(wav_files)}件")

for f in wav_files:
    print(f"ダウンロード: {f}")
    files.download(f)